In [0]:
# %sql
# DROP TABLE IF EXISTS vendor_status.silver.components;
# DROP TABLE IF EXISTS vendor_status.silver.incident_events;

In [0]:
from pyspark.sql.functions import explode, col

silver_checkpoint = "s3://vendor-status-monitor/checkpoints/silver/components/"
silver_table = "vendor_status.silver.components"

silver_components_df = (
    spark.readStream
    .table("vendor_status.bronze.status_checks")
    .select(
        col("page.name").alias("vendor"),
        explode("components").alias("c"),
        col("_ingested_at"),
        col("_bronze_loaded_at"),
        col("_source_file"),
    )
    .select(
        "vendor",
        col("c.id").alias("component_id"),
        col("c.name").alias("component_name"),
        col("c.status").alias("status"),
        col("c.group").alias("is_group"),
        col("c.group_id").alias("group_id"),
        col("_ingested_at").alias("observed_at"),
        "_bronze_loaded_at",
        "_source_file",
    )
)

(
    silver_components_df.writeStream
    .format("delta")
    .option("checkpointLocation", silver_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(silver_table)
)

In [0]:
from pyspark.sql.functions import col

page_status_checkpoint = "s3://vendor-status-monitor/checkpoints/silver/page_status/"
page_status_table = "vendor_status.silver.page_status"

page_status_df = (
    spark.readStream
    .table("vendor_status.bronze.status_checks")
    .select(
        col("page.name").alias("vendor"),
        col("status.indicator").alias("indicator"),
        col("status.description").alias("description"),
        col("_ingested_at").alias("observed_at"),
        "_bronze_loaded_at",
        "_source_file",
    )
)

(
    page_status_df.writeStream
    .format("delta")
    .option("checkpointLocation", page_status_checkpoint)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(page_status_table)
)

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.silver.component_changes AS
WITH ordered AS (
  SELECT
    vendor,
    component_id,
    component_name,
    status,
    observed_at,
    LAG(status) OVER (
      PARTITION BY vendor, component_id
      ORDER BY observed_at
    ) AS previous_status
  FROM vendor_status.silver.components
)
SELECT
  vendor,
  component_id,
  component_name,
  previous_status,
  status AS new_status,
  observed_at AS changed_at
FROM ordered
WHERE previous_status IS NOT NULL
  AND status <> previous_status
ORDER BY changed_at DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.silver.incidents AS
WITH exploded AS (
  SELECT
    page.name AS vendor,
    'incident' AS source_type,
    inc.id AS incident_id,
    inc.name,
    inc.status,
    inc.impact,
    inc.created_at,
    inc.updated_at,
    _bronze_loaded_at
  FROM vendor_status.bronze.status_checks
  LATERAL VIEW explode(incidents) AS inc

  UNION ALL

  SELECT
    page.name AS vendor,
    'scheduled_maintenance' AS source_type,
    inc.id AS incident_id,
    inc.name,
    inc.status,
    inc.impact,
    inc.created_at,
    inc.updated_at,
    _bronze_loaded_at
  FROM vendor_status.bronze.status_checks
  LATERAL VIEW explode(scheduled_maintenances) AS inc
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY vendor, source_type, incident_id
      ORDER BY updated_at DESC, _bronze_loaded_at DESC
    ) AS rn
  FROM exploded
)
SELECT vendor, source_type, incident_id, name, status, impact, created_at, updated_at
FROM ranked
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.silver.incident_updates AS
WITH exploded AS (
  SELECT
    page.name AS vendor,
    'incident' AS source_type,
    inc.id AS incident_id,
    inc.updated_at AS incident_updated_at,
    inc.incident_updates,
    _bronze_loaded_at
  FROM vendor_status.bronze.status_checks
  LATERAL VIEW explode(incidents) AS inc

  UNION ALL

  SELECT
    page.name AS vendor,
    'scheduled_maintenance' AS source_type,
    inc.id AS incident_id,
    inc.updated_at AS incident_updated_at,
    inc.incident_updates,
    _bronze_loaded_at
  FROM vendor_status.bronze.status_checks
  LATERAL VIEW explode(scheduled_maintenances) AS inc
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY vendor, source_type, incident_id
      ORDER BY incident_updated_at DESC, _bronze_loaded_at DESC
    ) AS rn
  FROM exploded
),
latest_only AS (
  SELECT vendor, source_type, incident_id, incident_updates
  FROM ranked
  WHERE rn = 1
)
SELECT
  vendor,
  source_type,
  incident_id,
  upd.id AS update_id,
  upd.status,
  upd.body,
  upd.created_at
FROM latest_only
LATERAL VIEW explode(incident_updates) AS upd;

In [0]:
%sql
SELECT * FROM vendor_status.silver.incidents
WHERE vendor = 'DigitalOcean' AND source_type = 'scheduled_maintenance';

In [0]:
%sql
SELECT * FROM vendor_status.silver.incident_updates
WHERE incident_id = (
  SELECT incident_id FROM vendor_status.silver.incidents
  WHERE vendor = 'DigitalOcean' AND source_type = 'scheduled_maintenance'
  LIMIT 1
);